# Tahap 6 - Laporan Analitik

**Tujuan:** mengumpulkan semua hasil Tahap 2-5 menjadi satu laporan tertulis yang bisa dipakai untuk portofolio dan dibahas saat interview.

Aturan: **laporan dihasilkan dari file CSV hasil eksekusi**, bukan ditulis manual dari ingatan. Kalau angka di CSV berubah, laporan ikut berubah saat notebook ini dijalankan ulang.

Output:
- `laporan/06_laporan_analisis.md`
- `outputs/06_jawaban_pertanyaan.csv`
- `outputs/06_inventaris_grafik.csv`


In [ ]:
# Sel 1: Muat seluruh tabel hasil Tahap 2-5
import os
import pandas as pd

BASE = '/content'
OUT = os.path.join(BASE, 'outputs')
LAP = os.path.join(BASE, 'laporan')
os.makedirs(LAP, exist_ok=True)

def baca(nama, **kw):
    return pd.read_csv(os.path.join(OUT, nama), encoding='utf-8-sig', **kw)

prov    = baca('provenance.csv').iloc[0]
bersih  = baca('tiktok_clean.csv')
log     = baca('03_log_cleaning.csv')
dur     = baca('04_ringkasan_durasi.csv')
hari    = baca('04_ringkasan_hari.csv')
jam     = baca('04_ringkasan_jam.csv')
ht      = baca('04_ringkasan_hashtag.csv')
kat     = baca('04_ringkasan_kategori.csv')
lift    = baca('04_lift_interaksi_tinggi.csv')
fol     = baca('04_ringkasan_follower.csv')
top     = baca('04_top1pct.csv')
korel   = baca('04_korelasi.csv')
boot    = baca('05_bootstrap_median.csv')
uji     = baca('05_uji_durasi.csv')
sens    = baca('05_sensitivitas_top1pct.csv', index_col=0)
ambang  = baca('05_sensitivitas_ambang.csv')
ringkas = baca('05_ringkasan_sensitivitas.csv')

# Dua tabel jam/hari hanya menyimpan fraksi; persentase dihitung di sini.
hari['median_like_pct'] = (hari.median_like_rate * 100).round(2)
jam['median_like_pct'] = (jam.median_like_rate * 100).round(2)

def num(x):
    return f'{x:,.0f}'.replace(',', '.')

def pct(x):
    return f'{x*100:.2f}%'

def koma(x, n=2):
    return f'{x:.{n}f}'.replace('.', ',')

def id_str(s):
    return str(s).replace('.', ',')

def baris(df, kolom, kunci, nilai, ambil):
    return df.loc[df[kunci] == nilai, ambil].iloc[0]

print('Tabel terbaca:', 15, '| data bersih:', bersih.shape)


In [ ]:
# Sel 2: Ambil angka kunci dari tabel (semua dari kode)
median_teks = baris(ringkas, 'hasil', 'temuan', 'Median like_rate seluruh data', 'hasil')
d1530 = dur[dur.duration_group == '15-30 detik'].iloc[0]
d60   = dur[dur.duration_group == '≥60 detik'].iloc[0]
d15   = dur[dur.duration_group == '<15 detik'].iloc[0]
selasa = hari[hari.post_day == 'Selasa'].iloc[0]
rabu   = hari[hari.post_day == 'Rabu'].iloc[0]
jam22  = jam[jam.post_hour == 22].iloc[0]
ht_sering = ht.nlargest(1, 'n_video').iloc[0]
ht_tinggi = ht.nlargest(1, 'median_like_pct').iloc[0]
fol_atas  = fol[fol.follower_tier == '≥1 jt'].iloc[0]
rho_fol   = baris(korel, 'spearman_rho', 'pasangan', 'author_followers vs like_rate', 'spearman_rho')
tahun_angka = pd.to_datetime(bersih.posted_at).dt.year
rho_tahun = round(tahun_angka.rank().corr(bersih.like_rate.rank()), 3)
top1 = top[top.grup == 'Top 1% views'].iloc[0]
rest = top[top.grup == 'Sisanya'].iloc[0]
n_top = int(top1.n_video)
idx_top = bersih.nlargest(n_top, 'views').index
kontrib = bersih.loc[idx_top, 'views'].sum() / bersih.views.sum() * 100
n_lain = int(kat[kat.kategori_heuristik == 'Lainnya'].n_video.iloc[0])
cakupan = (1 - n_lain / len(bersih)) * 100
n_mask = int(log[log.keputusan.str.contains('Tutup @akun')].jumlah_baris_terdampak.iloc[0])
n_log = len(log)
stabil_mentah = baris(ringkas, 'hasil', 'temuan', 'Item interaksi tinggi stabil', 'hasil')
stabil = [s.strip() for s in stabil_mentah.split(',')]
audio = [s for s in stabil if s.isdigit()]
tag_stabil = [s for s in stabil if not s.isdigit()]
tahun_stabil = baris(ringkas, 'hasil', 'temuan', 'Efek durasi stabil antar tahun', 'hasil')
tahan_outlier = baris(ringkas, 'hasil', 'temuan', 'Tahan terhadap pengecualian top 1%', 'hasil')
sig1530 = baris(uji, 'signifikan_0.05', 'perbandingan', '15-30 detik vs ≥60 detik', 'signifikan_0.05')

print('Angka kunci:', median_teks, '|', d1530.median_like_pct, d60.median_like_pct,
      '| top1%:', n_top, 'video', round(kontrib, 1), '%', '| rho tahun:', rho_tahun)


In [ ]:
# Sel 3: Tabel jawaban 8 pertanyaan -> CSV
jawaban = pd.DataFrame([
    {'no': 1,
     'pertanyaan': 'Jenis konten dengan views/likes tertinggi',
     'status': 'Sebagian',
     'ringkasan_jawaban': f'Hashtag paling sering {ht_sering.hashtag} pada {int(ht_sering.n_video)} video; like rate tertinggi {ht_tinggi.hashtag} {koma(ht_tinggi.median_like_pct)}% (n={int(ht_tinggi.n_video)})',
     'batasan': 'Tanpa comments/shares; kategori buatan sendiri hanya menjangkau ' + f'{cakupan:.1f}% video'},
    {'no': 2,
     'pertanyaan': 'Durasi vs tingkat interaksi',
     'status': 'Sebagian',
     'ringkasan_jawaban': f'15-30 detik {koma(d1530.median_like_pct)}% vs ≥60 detik {koma(d60.median_like_pct)}%; signifikan={bool(sig1530)}, tapi hanya {tahun_stabil} yang konsisten',
     'batasan': 'like_rate adalah proxy; pola tidak konsisten antar tahun'},
    {'no': 3,
     'pertanyaan': 'Jam & hari posting',
     'status': 'Bisa dengan syarat',
     'ringkasan_jawaban': f'Selasa {koma(selasa.median_like_pct)}% (n={int(selasa.n_video)}), Rabu {koma(rabu.median_like_pct)}% (n={int(rabu.n_video)}); jam 22 n={int(jam22.n_video)} like rate {koma(jam22.median_like_pct)}%',
     'batasan': 'Zona waktu tidak ada; jam 22 diduga artefak pengumpulan, bukan rekomendasi'},
    {'no': 4,
     'pertanyaan': 'Hashtag, sound, kata kunci caption',
     'status': 'Sebagian',
     'ringkasan_jawaban': f'Item stabil di beberapa ambang: {len(audio)} ID audio anonim + {", ".join(tag_stabil)}; kata caption tidak ada yang mencapai ambang 5 video',
     'batasan': 'Asosiasi bukan sebab-akibat; caption sudah disamarkan'},
    {'no': 5,
     'pertanyaan': 'Follower vs engagement rate',
     'status': 'Sebagian',
     'ringkasan_jawaban': f'Tier ≥1 jt follower like rate {koma(fol_atas.median_like_pct)}% (n={int(fol_atas.n_video)}); Spearman followers vs like_rate {koma(rho_fol, 3)}',
     'batasan': 'Engagement rate definisi awal tidak bisa dihitung'},
    {'no': 6,
     'pertanyaan': 'Perbedaan bahasa/wilayah, fokus Indonesia',
     'status': 'Tidak bisa dijawab',
     'ringkasan_jawaban': 'Tidak dijawab',
     'batasan': 'Tidak ada kolom bahasa, negara, atau region'},
    {'no': 7,
     'pertanyaan': 'Jenis interaksi dominan per konten',
     'status': 'Tidak bisa dijawab',
     'ringkasan_jawaban': 'Tidak dijawab',
     'batasan': 'Tidak ada kolom comments dan shares'},
    {'no': 8,
     'pertanyaan': 'Rekomendasi untuk kreator/brand',
     'status': 'Bisa dengan keterbatasan',
     'ringkasan_jawaban': 'Uji eksperimen format 15-30 detik dan hashtag ber-like-rate tinggi; jangan pakai jam posting sebagai rekomendasi',
     'batasan': 'Bukan bukti sebab-akibat; perlu validasi pada data baru'},
])
jawaban.to_csv(os.path.join(OUT, '06_jawaban_pertanyaan.csv'), index=False,
               encoding='utf-8-sig')
print(jawaban[['no', 'status']].to_string(index=False))


In [ ]:
# Sel 4: Inventaris grafik -> CSV (judul, sumbu, interpretasi)
grafik = pd.DataFrame([
 {'file':'04_distribusi_views.png','judul':'Distribusi Views (Skala Log10)','sumbu_x':'log10(views)','sumbu_y':'Jumlah video','interpretasi':'Mean tertarik ke kanan oleh sedikit video viral, sehingga median lebih mewakili video umum.'},
 {'file':'04_durasi_vs_like_rate.png','judul':'Like Rate Median per Kelompok Durasi','sumbu_x':'Kelompok durasi','sumbu_y':'Like rate median (%)','interpretasi':'Durasi 15-30 detik memiliki like rate median tertinggi sekaligus median views tertinggi di antara kelompok berdurasi jelas.'},
 {'file':'04_hari_posting.png','judul':'Like Rate Median per Hari Posting','sumbu_x':'Hari posting','sumbu_y':'Like rate median (%)','interpretasi':'Rabu punya like rate median terendah dengan sampel terbanyak, sedangkan Selasa tertinggi dengan sampel lebih sedikit.'},
 {'file':'04_jam_posting.png','judul':'Like Rate Median per Jam Posting','sumbu_x':'Jam posting (zona waktu tidak tercantum)','sumbu_y':'Like rate median (%)','interpretasi':'Jam 22 menyumbang sampel paling banyak dengan like rate rendah, kemungkinan artefak pengumpulan data.'},
 {'file':'04_top_hashtag.png','judul':'Top 10 Hashtag berdasarkan Like Rate Median (n >= 5)','sumbu_x':'Like rate median (%)','sumbu_y':'Hashtag','interpretasi':'Hashtag dengan like rate tinggi belum tentu paling sering dipakai.'},
 {'file':'04_kategori_konten.png','judul':'Like Rate Median per Kategori Konten (Heuristik Kata Kunci)','sumbu_x':'Like rate median (%)','sumbu_y':'Kategori (dari hashtag + caption)','interpretasi':'Kategori ini tesaurus buatan sendiri dan cakupannya terbatas, bukan label resmi TikTok.'},
 {'file':'04_follower_vs_like_rate.png','judul':'Like Rate Median per Tingkat Follower','sumbu_x':'Jumlah follower akun','sumbu_y':'Like rate median (%)','interpretasi':'Korelasi followers dengan like_rate lemah meski tier teratas punya median tertinggi.'},
 {'file':'04_top1pct_vs_sisanya.png','judul':'Perbandingan 1% Video Teratas (berdasarkan views) vs Sisanya','sumbu_x':'Grup','sumbu_y':'Views (median) dan like rate median (%)','interpretasi':'Delapan video teratas menyumbang porsi besar total views dengan like rate median sedikit lebih tinggi.'},
 {'file':'05_bootstrap_durasi.png','judul':'Median Like Rate per Durasi dengan Interval Kepercayaan Bootstrap 95%','sumbu_x':'Like rate median (%)','sumbu_y':'Kelompok durasi','interpretasi':'Interval 15-30 detik seluruhnya di atas interval ≥60 detik, jadi perbedaan sulit dijelaskan kebetulan sampel.'},
 {'file':'05_sensitivitas_top1pct.png','judul':'Sensitivitas Like Rate Median: Data Lengkap vs Tanpa 1% Video Teratas','sumbu_x':'Kelompok durasi','sumbu_y':'Like rate median (%)','interpretasi':'Membuang delapan video viral mengubah median maksimal 0,06 poin persen.'},
])
grafik.to_csv(os.path.join(OUT, '06_inventaris_grafik.csv'), index=False,
              encoding='utf-8-sig')
print('Grafik tercatat:', len(grafik))


In [ ]:
# Sel 5: Tulis laporan markdown dari angka yang sudah diambil
L = []
A = L.append

A('# Laporan Analisis Konten TikTok')
A('')
A('**Pertanyaan utama:** jenis konten apa yang paling banyak mendapat interaksi?')
A('')
A('Laporan portofolio Data Analyst. Semua angka diambil dari file CSV hasil eksekusi notebook Tahap 2-5. '
  'Jika data diubah dan notebook dijalankan ulang, angka di sini ikut berubah.')
A('')
A('## 1. Ringkasan eksekutif')
A('')
A(f'- Dataset **{prov.judul_dataset}**, lisensi **{prov.lisensi}**, diambil dengan **upload manual** dari Kaggle.')
A(f'- Setelah cleaning: **{len(bersih):,} baris × {bersih.shape[1]} kolom**; {n_log} keputusan cleaning tercatat.')
A(f'- Like rate median seluruh data **{id_str(median_teks)}**; like rate adalah proxy karena kolom komentar dan share tidak ada.')
A(f'- Kelompok durasi **15-30 detik** punya like rate median **{koma(d1530.median_like_pct)}%**, '
  f'lebih tinggi daripada **≥60 detik** **{koma(d60.median_like_pct)}%**; uji peringkat tetap signifikan setelah koreksi.')
A(f'- Namun pola durasi hanya konsisten pada **{tahun_stabil}** yang cukup sampel, jadi klaimnya dibatasi.')
A(f'- **{n_top} video teratas** menyumbang **{koma(kontrib, 1)}% total views**; '
  f'like rate median mereka {koma(top1.median_like_pct)}% vs {koma(rest.median_like_pct)}% pada sisanya.')
A(f'- Pertanyaan tentang bahasa/wilayah dan jenis interaksi dominan **tidak bisa dijawab** karena kolomnya tidak ada.')
A('')

A('## 2. Sumber data dan etika')
A('')
A(f'| Item | Keterangan |')
A(f'|---|---|')
A(f'| Judul dataset | {prov.judul_dataset} |')
A(f'| URL Kaggle | {prov.url_kaggle} |')
A(f'| Lisensi | {prov.lisensi} |')
A(f'| Cara pengumpulan | {prov.cara_pengumpulan} |')
A(f'| Rentang waktu | {prov.rentang_tanggal} |')
A(f'| Metode akuisisi | Upload manual, tanpa Kaggle API |')
A('')
A(f'Dataset ini adalah **snapshot**, bukan data real-time. Deskripsi Kaggle menyebut metadata video dari berbagai topik dan wilayah, '
  f'tetapi tidak ada kolom wilayah di file, jadi analisis Indonesia tidak mungkin dilakukan.')
A('')
A(f'Privasi: **{n_mask} caption** disamarkan (tanda @, label kontak, nomor diganti penanda). '
  f'Tidak ada nama akun, URL profil, atau kolom identitas yang disimpan. Caption mentah tidak dikutip di laporan ini. '
  f'Penyamaran memakai pola, jadi nama orang yang tidak mengikuti pola mungkin tetap tertinggal — ini keterbatasan yang jujur.')
A('')

A('## 3. Metode singkat (bahan interview)')
A('')
A('- **Median, bukan mean:** distribusi views sangat miring (beberapa video sangat viral), sehingga mean tidak mewakili video umum.')
A('- **like_rate = likes / views:** proxy interaksi. Definisi engagement_rate `(likes+comments+shares)/views` tidak bisa dihitung '
  'karena `comments` dan `shares` tidak tersedia. Kolom palsu tidak dibuat.')
A('- **Outlier tidak dihapus:** video viral justru menjadi objek analisis. Penanggulangannya memakai median, persentil, log skala, '
  'perbandingan top 1%, dan uji berbasis peringkat.')
A('- **Bootstrap 2.000 kali (seed 42):** mengukur rentang ketidakpastian median tanpa asumsi distribusi normal.')
A('- **Kruskal-Wallis dan Mann-Whitney + Bonferroni:** uji beda antar kelompok berbasis peringkat, dengan koreksi karena tiga pasangan diuji.')
A('- **Ambang frekuensi:** hashtag/sound minimal 5 video agar tidak didominasi satu video; klaim jam hanya untuk n ≥ 10.')
A('- **Kategori konten:** tesaurus kata kunci buatan sendiri, bukan label resmi platform.')
A('')

A('## 4. Temuan utama')
A('')
A('### 4.1 Distribusi dan video paling viral')
A('')
A(f'- Median views **{num(bersih.views.median())}**, mean **{num(bersih.views.mean())}**. '
  f'Perbedaan ini menunjukkan distribusi yang sangat condong.')
A(f'- {n_top} video teratas menguasai **{koma(kontrib, 1)}% total views** dan '
  f'{koma(bersih.loc[idx_top, "likes"].sum()/bersih.likes.sum()*100, 1)}% total likes.')
A('- Interpretasi: sebagian kecil video menyumbang porsi besar tayangan, jadi ringkasan harus memakai median.')
A('')
A('### 4.2 Durasi')
A('')
A(f'| Durasi | n | Median views | Like rate median | CI bootstrap |')
A('|---|---:|---:|---:|---|')
for _, r in dur.iterrows():
    b = boot[boot.durasi == r.duration_group]
    ci = f'{koma(b.ci_bawah_pct.iloc[0])}-{koma(b.ci_atas_pct.iloc[0])}%' if len(b) else '-'
    A(f'| {r.duration_group} | {int(r.n_video)} | {num(r.median_views)} | {koma(r.median_like_pct)}% | {ci} |')
A('')
A(f'- Uji Kruskal-Wallis dan perbandingan 15-30 vs ≥60 detik tetap signifikan setelah Bonferroni.')
A(f'- Sensitivitas outlier: {id_str(tahan_outlier)}; Spearman durasi-like_rate '
  f'{koma(baris(korel, "spearman_rho", "pasangan", "duration_sec vs like_rate", "spearman_rho"), 3)}.')
A(f'- Kontrol follower: pola 15-30 > ≥60 konsisten di semua tier. Kontrol tahun: hanya **{tahun_stabil}**.')
A('- Interpretasi: durasi pendek berasosiasi dengan like_rate lebih tinggi pada sampel ini, tetapi efek tahun belum terkontrol penuh; '
  'jangan dinyatakan sebagai sebab-akibat.')
A('')
A('### 4.3 Waktu posting')
A('')
A(f'- Hari: Selasa **{koma(selasa.median_like_pct)}%** (n={int(selasa.n_video)}), '
  f'Rabu **{koma(rabu.median_like_pct)}%** (n={int(rabu.n_video)}).')
A(f'- Jam 22:00 punya **{int(jam22.n_video)} video** dengan like rate **{koma(jam22.median_like_pct)}%**.')
A('- Interpretasi: konsentrasi sampel pada jam tertentu dan tidak adanya zona waktu membuat hasil ini tidak layak dijadikan '
  'rekomendasi jam posting; kemungkinan besar artefak pengumpulan data.')
A('')
A('### 4.4 Hashtag, audio, dan kata kunci')
A('')
A(f'- Paling sering: **{ht_sering.hashtag}** pada {int(ht_sering.n_video)} video '
  f'(like rate {koma(ht_sering.median_like_pct)}%).')
A(f'- Like rate tertinggi (n ≥ 5): **{ht_tinggi.hashtag}** **{koma(ht_tinggi.median_like_pct)}%** '
  f'(n={int(ht_tinggi.n_video)}).')
A(f'- Item yang muncul di lebih dari satu definisi ambang: **{len(audio)} ID audio anonim** dan '
  f'**{", ".join(tag_stabil)}**.')
A('- Uji kata kunci caption dari daftar heuristik tidak menemukan kata yang muncul di minimal 5 video grup atas, '
  'jadi tidak dibuat klaim.')
A('- Interpretasi: lift di atas 1 berarti item lebih sering muncul di kelompok like_rate tinggi; '
  'ini asosiasi, bukan bukti bahwa item itu menyebabkan like.')
A('')
A('### 4.5 Follower')
A('')
A(f'- Tier **≥1 jt follower**: like rate **{koma(fol_atas.median_like_pct)}%** (n={int(fol_atas.n_video)}), '
  f'median views {num(fol_atas.median_views)}.')
A(f'- Spearman followers vs like_rate **{koma(rho_fol, 3)}**, followers vs views '
  f'**{koma(baris(korel, "spearman_rho", "pasangan", "author_followers vs views", "spearman_rho"), 3)}**.')
A('- Interpretasi: akun besar punya tayangan lebih banyak, tetapi like per tayangan tidak ikut naik kuat.')
A('')
A('### 4.6 Kategori heuristik')
A('')
A(f'- Kategori buatan sendiri hanya menjangkau **{cakupan:.1f}%** video; sisanya ({n_lain} video) masuk **Lainnya**.')
A('- Interpretasi: ini contoh metode pengelompokan, bukan kesimpulan utama dan bukan label resmi platform.')
A('')

A('## 5. Jawaban 8 pertanyaan')
A('')
A('| # | Pertanyaan | Status | Jawaban ringkas |')
A('|---|---|---|---|')
for _, r in jawaban.iterrows():
    A(f'| {r["no"]} | {r["pertanyaan"]} | {r["status"]} | {r["ringkasan_jawaban"]} |')
A('')
A('Detail batasan setiap jawaban ada di `outputs/06_jawaban_pertanyaan.csv`.')
A('')

A('## 6. Keterbatasan')
A('')
A('- Tidak ada `comments` dan `shares`, jadi engagement rate definisi awal tidak bisa dihitung.')
A('- Tidak ada kolom bahasa, negara, wilayah, atau usia penonton. Tidak ada klaim tentang Gen Z, anak, orang tua, atau Indonesia.')
A(f'- Data snapshot lintas 2015-2025; video lama punya waktu lebih lama mengumpulkan views '
  f'(Spearman tahun vs like_rate {koma(rho_tahun, 3)}).')
A('- Zona waktu tidak tercantum, sehingga jam posting tidak bisa diartikan secara lokal.')
A(f'- Kategori konten heuristik hanya menjangkau {cakupan:.1f}% video.')
A('- Penyamaran caption berbasis pola, bukan jaminan 100% tanpa nama orang.')
A('- Semua hubungan bersifat asosiasi; tidak ada desain eksperimen, jadi tidak ada klaim sebab-akibat.')
A('- Cara pengumpulan dataset tidak dijelaskan rinci di halaman Kaggle; tidak saya karang.')
A('')

A('## 7. Daftar grafik')
A('')
A('| File | Judul | Sumbu X | Sumbu Y | Interpretasi |')
A('|---|---|---|---|---|')
for _, r in grafik.iterrows():
    A(f'| `{r["file"]}` | {r["judul"]} | {r["sumbu_x"]} | {r["sumbu_y"]} | {r["interpretasi"]} |')
A('')
A('Inventaris ini juga tersimpan di `outputs/06_inventaris_grafik.csv`. '
  'Dashboard tidak dibuat dalam sesi ini.')
A('')

A('## 8. Cara menjalankan')
A('')
A('```python')
A('# Di Google Colab, unggah folder project, lalu jalankan berurutan:')
A('# 01_setup.ipynb -> 02_data_understanding.ipynb -> 03_data_cleaning.ipynb')
A('# -> 04_eda.ipynb -> 05_analisis_lanjutan.ipynb -> 06_laporan_analitik.ipynb')
A('')
A("# Pastikan BASE sesuai folder kerja (di Colab: /content) dan folder data/ berisi tiktok_data.csv.")
A('```')
A('')
A('Struktur folder: `data/` (dataset mentah, tidak di-commit), `notebooks/` (kode), '
  '`outputs/` (tabel hasil), `images/` (grafik), `laporan/` (laporan ini).')
A('')

isi = '\n'.join(L) + '\n'
path = os.path.join(LAP, '06_laporan_analisis.md')
with open(path, 'w', encoding='utf-8') as f:
    f.write(isi)
print('Laporan ditulis:', path, '| baris:', len(L), '| karakter:', len(isi))


In [ ]:
# Sel 6: Cek cepat isi laporan
isi = open(os.path.join(LAP, '06_laporan_analisis.md'), encoding='utf-8').read()
print('Ada angka 4,84%:', '4,84%' in isi or '4.84%' in isi)
print('Ada keterangan tidak bisa dijawab:', 'Tidak bisa dijawab' in isi)
print('Ada daftar grafik:', len(grafik), 'entri')
print('Jumlah tabel jawaban:', len(jawaban))
print(isi[:400])
